# **MÓDULO 15 - Exercício**
# Análise  - A segunda etapa da Pré Modelagem

# 1) O primeiro exercício é o de salvar a base que criaram na atividade do módulo anterior em csv e abrir ela neste arquivo.
Igual fizemos no início do módulo atual no início da primeira aula.

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.express as px
from narwhals import read_csv
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Seu código de importação aqui
df = pd.read_csv("CHURN_TELECON_TRATADO.csv")
df.head()

,customerID,Genero,Idoso,Casado,Dependentes,Tempo_como_Cliente,Servico_Telefonico,Servico_Internet,Servico_Seguranca,Suporte_Tecnico,StreamingTV,Tipo_Contrato,Forma_Pagamento,Pagamento_Mensal,Total_Pago,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,DSL,No,No,No,Month-to-month,Electronic check,29.850000,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,DSL,Yes,No,No,One year,Mailed check,56.950000,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,DSL,Yes,No,No,Month-to-month,Mailed check,53.850000,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,DSL,Yes,Yes,No,One year,Bank transfer (automatic),40.905556,1840.75,No
4,8191-XWSZG,Female,0,No,No,52,Yes,No,No internet service,No internet service,No internet service,One year,Mailed check,19.672115,1022.95,No


## Transformações nos Dados

- **Codificação de variáveis binárias**: converti os valores `'no'` para `0` e `'yes'` para `1`.
- **Ajuste de tipos de dados**: alterei os tipos das colunas para garantir coerência e eficiência no armazenamento.
- **Remoção de coluna**: excluí a coluna `serviço telefônico` devido à alta quantidade de valores nulos. Embora tenha tratado esses dados anteriormente, percebi que essa variável não agregaria valor à análise atual.

In [3]:
df = df.replace({'No': 0,'Yes': 1})
df = df.replace({'No internet service': 0})
df = df.drop(columns=['Servico_Telefonico'])

colunas = ['Servico_Seguranca','Suporte_Tecnico','StreamingTV','Casado', 'Dependentes','Churn' ]

df[colunas] = df[colunas].astype(int)

In [4]:
print(df.dtypes)

customerID                str
Genero                    str
Idoso                   int64
Casado                  int64
Dependentes             int64
Tempo_como_Cliente      int64
Servico_Internet       object
Servico_Seguranca       int64
Suporte_Tecnico         int64
StreamingTV             int64
Tipo_Contrato             str
Forma_Pagamento           str
Pagamento_Mensal      float64
Total_Pago            float64
Churn                   int64
dtype: object


# 2) Comece pela análise univariada:

A) Utilize a função describe no seu dataframe, veja os insights que consegue retirar.

B) Já é possível identificar variáveis com possíveis outliers? Se sim, quais?

C) Plot gráficos que considerar importante para completar sua análise univariada. (Lembrando que sua variável preditora é o churn). Não se esqueça de trazer insights de cada gráfico plotado. Utilize pelo menos 4 variáveis distintas.

D) Verifique se os dados das variáveis Booleanas são balanceados ou não.

In [5]:
print(df.describe())

             Idoso       Casado  Dependentes  Tempo_como_Cliente  \
count  2488.000000  2488.000000  2488.000000         2488.000000   
mean      0.161576     0.492765     0.314711           32.352090   
std       0.368135     0.500048     0.464494           24.636885   
min       0.000000     0.000000     0.000000            0.000000   
25%       0.000000     0.000000     0.000000            8.000000   
50%       0.000000     0.000000     0.000000           29.000000   
75%       0.000000     1.000000     1.000000           56.000000   
max       1.000000     1.000000     1.000000           72.000000   

       Servico_Seguranca  Suporte_Tecnico  StreamingTV  Pagamento_Mensal  \
count        2488.000000      2488.000000  2488.000000       2488.000000   
mean            0.284164         0.285772     0.385852         65.511708   
std             0.451106         0.451872     0.486894         29.920953   
min             0.000000         0.000000     0.000000         15.725000   
25%    

## Identificação de Variáveis Booleanas

As seguintes colunas foram identificadas como variáveis booleanas e já foram padronizadas:

- `Idoso`
- `Casado`
- `Dependentes`
- `Servico_Seguranca`
- `Suporte_Tecnico`
- `StreamingTV`
- `Churn` (variável-alvo)

## Identificação de Outliers

In [6]:
colunas_numericas = [
    'Tempo_como_Cliente',
    'Pagamento_Mensal',
    'Total_Pago'
]

for coluna in colunas_numericas:
    fig = px.box(
        df,
        y=coluna,
        points='outliers',
        title=f'Boxplot Interativo - {coluna}',
        labels={coluna: coluna}
    )

    fig.show()

In [10]:
colunas_numericas = [
    'Tempo_como_Cliente',
    'Pagamento_Mensal',
    'Total_Pago'
]

for coluna in colunas_numericas:
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = df[
        (df[coluna] < limite_inferior) |
        (df[coluna] > limite_superior)
    ]

    print(f'\nColuna: {coluna}')
    print(f'Q1: {Q1}')
    print(f'Q3: {Q3}')
    print(f'IQR: {IQR}')
    print(f'Limite inferior: {limite_inferior}')
    print(f'Limite superior: {limite_superior}')
    print(f'Quantidade de outliers: {len(outliers)}')


Coluna: Tempo_como_Cliente
Q1: 8.0
Q3: 56.0
IQR: 48.0
Limite inferior: -64.0
Limite superior: 128.0
Quantidade de outliers: 0

Coluna: Pagamento_Mensal
Q1: 39.6375
Q3: 90.25
IQR: 50.6125
Limite inferior: -36.281249999999986
Limite superior: 166.16875
Quantidade de outliers: 0

Coluna: Total_Pago
Q1: 402.3375
Q3: 3867.1625
IQR: 3464.825
Limite inferior: -4794.9
Limite superior: 9064.4
Quantidade de outliers: 0


Foram criados boxplots interativos para as variáveis Tempo_como_Cliente, Pagamento_Mensal e Total_Pago, com o objetivo de visualizar possíveis valores atípicos.

Em seguida, os resultados foram confirmados por meio do cálculo do intervalo interquartil (IQR). A análise mostrou que nenhuma das três variáveis apresentou outliers segundo o critério de
1,5
×
𝐼
𝑄
𝑅
1,5×IQR.

In [11]:
colunas_booleanas = [
    'Servico_Seguranca',
    'Suporte_Tecnico',
    'StreamingTV',
    'Casado',
    'Dependentes',
    'Churn'
]

for coluna in colunas_booleanas:
    contagem = (
        df[coluna]
        .value_counts()
        .reindex([0, 1], fill_value=0)
        .rename_axis('Valor')
        .reset_index(name='Quantidade')
    )

    contagem['Categoria'] = contagem['Valor'].map({
        0: 'Não',
        1: 'Sim'
    })

    fig = px.bar(
        contagem,
        x='Categoria',
        y='Quantidade',
        text='Quantidade',
        color='Categoria',
        title=f'Distribuição da variável {coluna}',
        labels={
            'Categoria': coluna,
            'Quantidade': 'Quantidade de registros'
        },
        category_orders={
            'Categoria': ['Não', 'Sim']
        }
    )

    fig.update_traces(textposition='outside')
    fig.update_layout(showlegend=False)
    fig.show()

## Análise das variáveis booleanas

Foram analisadas as variáveis booleanas por meio de gráficos de barras.

As colunas `Servico_Seguranca` e `Suporte_Tecnico` apresentaram desbalanceamento, mas isso pode ser explicado pelo fato de que clientes sem serviço de internet não precisam contratar esses serviços.

As variáveis `StreamingTV`, `Dependentes` e `Churn` também estão desbalanceadas, pois existe uma diferença significativa entre as categorias `0` e `1`.

Esse desbalanceamento deve ser considerado nas análises futuras, principalmente na variável `Churn`, pois a quantidade de clientes que não cancelaram o serviço é maior do que a quantidade de clientes que cancelaram.

# 3) Identifique e trate as colunas que contém outliers.
Caso opte por mante-los ou altera-los justifique sua escolha.

## Identificação e tratamento de outliers

Foram analisadas as colunas `Tempo_como_Cliente`, `Pagamento_Mensal` e `Total_Pago` por meio de boxplots e do cálculo do intervalo interquartil (IQR).

A análise mostrou que nenhuma das colunas possui outliers segundo o critério de \(1{,}5 \times IQR\). Portanto, não foi necessário remover ou alterar nenhum valor da base de dados.

Os dados foram mantidos sem alterações para preservar as informações originais e evitar a modificação indevida dos registros.

# 4) Realize a etapa da análise bivariada:
A) Questione pelo menos 5 informações e traga as respostas utilizando visuais gráficos e insights.


B) Quais variáveis você acredita serem as mais importantes para esse projetos relacionadas a variável Churn?

## Planejamento da análise bivariada

Para realizar a análise bivariada, foram definidas cinco perguntas sobre possíveis relações entre as características dos clientes e o `Churn`:

1. Existe relação entre ter dependentes e o `Churn`?
2. Pessoas casadas apresentam uma taxa maior de `Churn`?
3. O tempo como cliente influencia o cancelamento?
4. O valor do pagamento mensal influencia o `Churn`?
5. Será feita após a primeira análise.

Para responder às duas primeiras perguntas, serão utilizados **gráficos de barras agrupadas**. Esse tipo de gráfico permite comparar as categorias de cada variável com a quantidade de clientes que cancelaram ou permaneceram no serviço.

Para as perguntas relacionadas às variáveis numéricas, serão utilizados **boxplots**, possibilitando comparar a distribuição do tempo como cliente e do pagamento mensal entre os clientes que cancelaram e os que permaneceram.

A seguir, serão apresentados os códigos utilizados para gerar os gráficos e analisar essas relações.

In [12]:
# Dependentes x Churn
fig = px.histogram(
    df,
    x='Dependentes',
    color='Churn',
    barmode='group',
    text_auto=True,
    title='Relação entre Dependentes e Churn',
    labels={
        'Dependentes': 'Possui dependentes',
        'Churn': 'Cancelou o serviço',
        'count': 'Quantidade de clientes'
    }
)

fig.show()

### 1. Dependentes e Churn
A análise indica que não existe uma relação forte entre possuir dependentes e o Churn. A quantidade de cancelamentos permanece relativamente baixa tanto entre os clientes com dependentes quanto entre aqueles sem dependentes.

In [13]:
# Casado x Churn
fig = px.histogram(
    df,
    x='Casado',
    color='Churn',
    barmode='group',
    text_auto=True,
    title='Relação entre Estado Civil e Churn',
    labels={
        'Casado': 'É casado',
        'Churn': 'Cancelou o serviço',
        'count': 'Quantidade de clientes'
    }
)

fig.show()

### 2. Estado civil e Churn
Observa-se que a porcentagem de clientes casados que cancelaram o serviço é menor. Isso pode indicar que clientes casados apresentam menor tendência ao cancelamento quando comparados aos clientes não casados.

In [14]:
# Tempo como cliente x Churn
fig = px.box(
    df,
    x='Churn',
    y='Tempo_como_Cliente',
    color='Churn',
    points='all',
    title='Tempo como Cliente por Situação de Churn',
    labels={
        'Churn': 'Cancelou o serviço',
        'Tempo_como_Cliente': 'Tempo como cliente'
    }
)

fig.show()

### 3. Tempo como cliente e Churn
Os resultados sugerem que quanto maior o tempo de relacionamento com a empresa, menor é a probabilidade de cancelamento. Clientes recentes parecem apresentar maior risco de Churn, enquanto clientes antigos demonstram maior retenção.

In [15]:
# Pagamento mensal x Churn
fig = px.box(
    df,
    x='Churn',
    y='Pagamento_Mensal',
    color='Churn',
    points='all',
    title='Pagamento Mensal por Situação de Churn',
    labels={
        'Churn': 'Cancelou o serviço',
        'Pagamento_Mensal': 'Pagamento mensal'
    }
)

fig.show()

In [16]:
df['Faixa_Pagamento'] = pd.cut(
    df['Pagamento_Mensal'],
    bins=4,
    labels=[
        'Baixo',
        'Médio-baixo',
        'Médio-alto',
        'Alto'
    ]
)

fig = px.histogram(
    df,
    x='Faixa_Pagamento',
    color='Churn',
    barmode='group',
    text_auto=True,
    title='Churn por Faixa de Pagamento Mensal',
    labels={
        'Faixa_Pagamento': 'Faixa de pagamento mensal',
        'Churn': 'Cancelou o serviço',
        'count': 'Quantidade de clientes'
    }
)

fig.show()


### 4. Pagamento mensal e Churn
Essa foi uma das relações que merece maior atenção. A análise das medianas indica que clientes com pagamentos mensais mais altos apresentam maior probabilidade de cancelar o serviço. Esse resultado pode indicar a necessidade de investigar os planos mais caros e avaliar estratégias como descontos, benefícios ou ofertas personalizadas para reduzir o cancelamento.

### 5. Pessoas casadas, dependentes e Churn
Para complementar a análise, será investigada a relação conjunta entre estado civil, dependentes e Churn. Essa análise permitirá verificar se clientes casados com dependentes apresentam um comportamento diferente em relação ao cancelamento.

In [18]:
grafico = (
    df.groupby(['Casado', 'Dependentes', 'Churn'])
      .size()
      .reset_index(name='Quantidade')
)

# Calcula o total de cada grupo de Casado e Dependentes
grafico['Total_Grupo'] = (
    grafico.groupby(['Casado', 'Dependentes'])['Quantidade']
           .transform('sum')
)

# Calcula a porcentagem de Churn dentro de cada grupo
grafico['Percentual'] = (
    grafico['Quantidade'] / grafico['Total_Grupo'] * 100
).round(2)

grafico['Casado'] = grafico['Casado'].map({
    0: 'Não casado',
    1: 'Casado'
})

grafico['Dependentes'] = grafico['Dependentes'].map({
    0: 'Sem dependentes',
    1: 'Com dependentes'
})

grafico['Churn'] = grafico['Churn'].map({
    0: 'Não cancelou',
    1: 'Cancelou'
})

fig = px.bar(
    grafico,
    x='Casado',
    y='Percentual',
    color='Churn',
    facet_col='Dependentes',
    barmode='group',
    text='Percentual',
    title='Percentual de Churn por Estado Civil e Dependentes',
    labels={
        'Casado': 'Estado civil',
        'Percentual': 'Percentual (%)',
        'Churn': 'Situação do cliente'
    }
)

fig.update_traces(
    texttemplate='%{text:.2f}%',
    textposition='outside'
)

fig.update_yaxes(ticksuffix='%')
fig.show()

### 5. Estado civil, dependentes e Churn

A análise mostra que **pessoas solteiras e sem dependentes apresentam a maior proporção de cancelamento**.

Por outro lado, os clientes **que possuem dependentes apresentam maior tendência a permanecer na empresa**, indicando uma menor proporção de `Churn` nesse grupo.

Para complementar essa análise, também será investigada a relação entre o estado civil e o valor do pagamento mensal, comparando os pagamentos de pessoas casadas e não casadas. Essa comparação pode ajudar a verificar se existe diferença nos valores pagos entre esses grupos.

In [21]:
df_analise = df.copy()

df_analise['Estado_Civil'] = df_analise['Casado'].map({
    0: 'Não casado',
    1: 'Casado'
})

df_analise['Situacao_Churn'] = df_analise['Churn'].map({
    0: 'Não cancelou',
    1: 'Cancelou'
})

fig = px.box(
    df_analise,
    x='Estado_Civil',
    y='Pagamento_Mensal',
    color='Situacao_Churn',
    points='all',
    title='Pagamento Mensal por Estado Civil e Churn',
    labels={
        'Estado_Civil': 'Estado civil',
        'Pagamento_Mensal': 'Pagamento mensal',
        'Situacao_Churn': 'Situação do cliente'
    }
)

fig.show()

### Relação entre estado civil e pagamento mensal

O gráfico indica que **clientes casados tendem a pagar um valor mensal um pouco maior** do que clientes não casados.

Essa diferença aparece tanto entre os clientes que cancelaram quanto entre os que permaneceram. Porém, os valores são parecidos e existe bastante sobreposição entre os grupos.

## Variáveis mais importantes para o Churn

No início das análises, percebi que o `Pagamento_Mensal` e a combinação entre estado civil e dependentes pareciam ter uma relação importante com o `Churn`. Clientes que pagam valores mensais mais altos apresentaram mais cancelamentos, e pessoas solteiras sem dependentes tiveram uma maior proporção de `Churn`.

Porém, enquanto estava corrigindo a atividade, percebi um erro na minha análise: eu não havia considerado a variável `Tipo_Contrato`. Depois de analisá-la, observei uma relação muito mais forte com o cancelamento. Os clientes com contrato `Month-to-month` apresentaram uma quantidade de cancelamentos muito maior do que os clientes com contratos de um ou dois anos.

Com isso, percebi que o tipo de contrato pode ser mais importante para entender o `Churn` do que algumas variáveis analisadas anteriormente. Para diminuir os cancelamentos, a empresa poderia incentivar os clientes a escolherem contratos mais longos, oferecendo descontos ou outros benefícios.

Mesmo assim, o `Pagamento_Mensal`, o estado civil e a quantidade de dependentes continuam sendo informações úteis para identificar clientes com maior risco de cancelamento.

**A seguir, será apresentada a análise dos tipos de contrato em relação ao `Churn`.**

In [22]:
df['Tipo_Contrato'].value_counts()

Tipo_Contrato
Month-to-month    1369
Two year           602
One year           517
Name: count, dtype: int64

In [23]:
analise_contrato = (
    df.groupby(['Tipo_Contrato', 'Churn'])
      .size()
      .reset_index(name='Quantidade')
)

analise_contrato['Total_Contrato'] = (
    analise_contrato
    .groupby('Tipo_Contrato')['Quantidade']
    .transform('sum')
)

analise_contrato['Percentual'] = (
    analise_contrato['Quantidade']
    / analise_contrato['Total_Contrato']
    * 100
).round(2)

analise_contrato['Situacao_Churn'] = analise_contrato['Churn'].map({
    0: 'Não cancelou',
    1: 'Cancelou'
})

fig = px.bar(
    analise_contrato,
    x='Tipo_Contrato',
    y='Percentual',
    color='Situacao_Churn',
    barmode='group',
    text='Percentual',
    title='Percentual de Churn por Tipo de Contrato',
    labels={
        'Tipo_Contrato': 'Tipo de contrato',
        'Percentual': 'Percentual (%)',
        'Situacao_Churn': 'Situação do cliente'
    }
)

fig.update_traces(
    texttemplate='%{text:.2f}%',
    textposition='outside'
)

fig.update_yaxes(ticksuffix='%')
fig.show()